### Semantic Chunking
- SemanticChunker is a document splitter that uses embedding similarity between sentences to decide chunk boundaries.

- It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character/token splitters.

---

### 💡 Interview & Learning Notes

**Key Interview Questions:**
1. *Why use Semantic Chunking instead of `RecursiveCharacterTextSplitter`?* 
   - Fixed-size chunkers (like character splitters) can split paragraphs right in the middle of a thought, separating a pronoun from its subject. Semantic chunking uses an embedding model to calculate the similarity between adjacent sentences. If the similarity drops below a threshold, it assumes the topic changed and creates a split there. This yields much higher quality RAG context.
2. *What is the downside of Semantic Chunking?* 
   - Cost and Time. It requires running an embedding model *during* the ingestion phase on every single sentence before the final chunks are even created. If you are using API-based embeddings like OpenAI, this gets very expensive very fast.

**Learning Takeaways:**
- Use a cheap, fast, local embedding model (like `all-MiniLM-L6-v2`) for the Semantic Chunking pass, and then use your high-quality API embeddings (like OpenAI) for the final document indexing pass.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
## Initialize the model
model=SentenceTransformer('all-MiniLM-L6-v2')

## Sample text
text="""
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

## Step 1 : Split into sentences
sentences=[s.strip() for s in text.split("\n") if s.strip()]

### sstep 2: Embed each setence
embeddings=model.encode(sentences)

# Step 3: Initialize parameters
threshold = 0.7  # control chunk tightness
chunks = []
current_chunk=[sentences[0]]

## Step 4: Semantic grouping based on threshold

for i in range(1, len(sentences)):
    sim = cosine_similarity(
        [embeddings[i - 1]],
        [embeddings[i]]
    )[0][0]

    if sim>=threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk=[sentences[i]]

# Append the last chunk
chunks.append(" ".join(current_chunk))

# Output the chunks
print("\n📌 Semantic Chunks:")
for idx, chunk in enumerate(chunks):
    print(f"\nChunk {idx+1}:\n{chunk}")






### RAG Pipeline Modular Coding

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter
import os

# Ensure API Key is loaded
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")


In [ ]:
### Custom Semantic Chunker With Threshold
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


class Document:
    """Mock Document class for compatibility with LangChain-style docs."""

    def __init__(self, page_content: str, metadata: dict = None):
        self.page_content = page_content
        self.metadata = metadata if metadata is not None else {}


class ThresholdSemanticChunker:

    def __init__(self, model_name: str = "all-mpnet-base-v2", threshold: float = 0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def split(self, text: str) -> list[str]:
        # Split by period and filter empty strings
        sentences = [s.strip() for s in text.split(".") if s.strip()]

        if not sentences:
            return []

        # Generate all embeddings at once
        embeddings = self.model.encode(sentences)
        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            # Reshape vectors to 2D arrays for sklearn's cosine_similarity
            vec1 = embeddings[i - 1].reshape(1, -1)
            vec2 = embeddings[i].reshape(1, -1)
            sim = cosine_similarity(vec1, vec2)[0][0]

            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk) + ".")
                current_chunk = [sentences[i]]  # Reset to a fresh chunk list

        # Append the final remaining chunk
        if current_chunk:
            chunks.append(". ".join(current_chunk) + ".")

        return chunks

    def split_documents(self, docs: list) -> list:
        result = []
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(
                    Document(page_content=chunk, metadata=doc.metadata.copy())
                )
        return result  # Indentation fixed to loop through all docs


In [ ]:
# Sample text
sample_text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [ ]:
### Chunking
chunker = ThresholdSemanticChunker(threshold=0.7)
chunks = chunker.split_documents([doc])
chunks



[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'),
 Document(metadata={}, page_content='You can create chains, agents, memory, and retrievers.'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [ ]:
### VectorStore
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever()


In [ ]:
## Prompt Template

template = """Answer the question based on the following context:

{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
prompt


PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\n')

In [ ]:
## LLM
llm = init_chat_model(model="groq:gemma2-9b-it", temperature=0.4)

### LCEL Chain With retrieval (Future-safe RunnableParallel & itemgetter)

rag_chain = (
    RunnableParallel(
        {
            "context": itemgetter("question") | retriever,
            "question": itemgetter("question"),  
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

# --- Run Query ---
query = {"question": "What is LangChain used for?"}
result = rag_chain.invoke(query)

print(result)


According to the provided context, LangChain is a framework for building applications with LLMs. 



### Semantic chunker With Langchain

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader

# Component Explanation: SemanticChunker
# - Purpose: Splits documents dynamically based on sentence embedding similarity.
# - Breakpoint Strategies:
#   1. 'percentile' (default): Splits when distance exceeds the X-th percentile of distance differences.
#   2. 'standard_deviation': Splits when distance exceeds X standard deviations from the mean.
#   3. 'interquartile': Splits based on the Interquartile Range (IQR).
#   4. 'gradient': Splits based on gradient changes in distance.


In [ ]:
import os

# Ensure sample text file exists
file_path = "langchain_intro.txt"
if not os.path.exists(file_path):
    with open(file_path, "w", encoding="utf-8") as f:
        f.write("""LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.""")

## Load the documents
loader = TextLoader(file_path)
docs = loader.load()

## Initialize embedding model
embedding = OpenAIEmbeddings()

## Create the semantic chunker (configured with percentile threshold strategy)
chunker = SemanticChunker(
    embeddings=embedding,
    breakpoint_threshold_type="percentile",  # 'percentile', 'standard_deviation', 'interquartile', 'gradient'
    breakpoint_threshold_amount=95.0
)

## Split the documents
chunks = chunker.split_documents(docs)

## Result
for i, chunk in enumerate(chunks):
    print(f"\n chunk {i+1}:\n{chunk.page_content}")



 chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

 chunk 2:
You can create chains, agents, memory, and retrievers. The Eiffel Tower is located in Paris. France is a popular tourist destination.


### 🚀 Best Practices for Semantic Chunking

1. **Cost Efficiency & Token Optimization**:
   - Never use `text-embedding-3-large` or expensive API models for the `SemanticChunker`. Use a fast, free local HuggingFace model (`HuggingFaceEmbeddings`) for the chunking logic. Only use the expensive OpenAI model on the final `vectorstore.from_documents` ingestion.

2. **Time Optimization**:
   - Semantic chunking is slow. If processing millions of documents, use a standard `RecursiveCharacterTextSplitter` unless you specifically see retrieval accuracy issues caused by context shearing.

3. **Data Quality**:
   - Experiment with the threshold metrics (`percentile`, `standard_deviation`, `interquartile`). The default percentile method is usually the most robust across different document lengths.